# EDA — Precios de Propiedades Miami / Sur de Florida

Objetivo: entender la distribución del target, los features más relevantes y los patrones geográficos antes de modelar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

train = pd.read_csv('../data/tabular/train_processed.csv')
test  = pd.read_csv('../data/tabular/test_processed.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')

## 1. Resumen general

In [ ]:
train.describe(include='all').T[['count','mean','std','min','50%','max']]

In [ ]:
# Valores faltantes
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% de valores faltantes')
ax.set_title('Valores faltantes por columna (train)')
for i, v in enumerate(missing_pct):
    ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2. Distribución del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Escala original
axes[0].hist(train['lastSoldPrice_hpi_adjusted'], bins=60, color='steelblue', edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].set_title('Precio de venta (escala original)')
axes[0].set_xlabel('Precio ($)')

# Escala log
axes[1].hist(train['log_price'], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('log_price = log1p(precio)')
axes[1].set_xlabel('log1p(Precio)')

plt.suptitle('Distribución del target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(train['lastSoldPrice_hpi_adjusted'].describe().apply(lambda x: f'${x:,.0f}'))

## 3. Tipo de propiedad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Conteo
counts = train['homeType'].value_counts()
counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Cantidad por tipo de propiedad')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Precio mediano
medians = train.groupby('homeType')['lastSoldPrice_hpi_adjusted'].median().sort_values(ascending=False)
medians.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].set_title('Precio mediano por tipo de propiedad')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Features numéricos clave

In [ ]:
num_features = ['bedrooms', 'bathrooms', 'livingArea', 'yearBuilt', 
                'lotAreaValue', 'avg_school_rating', 'hoa_fee_monthly']

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()

for i, col in enumerate(num_features):
    data = train[col].dropna()
    # Limitar outliers extremos para visualización
    p99 = data.quantile(0.99)
    data = data[data <= p99]
    axes[i].hist(data, bins=40, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Distribución de features numéricos (sin outliers extremos)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Correlación con el precio

In [ ]:
exclude = ['zpid', 'lastSoldPrice_hpi_adjusted', 'log_price', 'zipcode', 'zip_3digit']
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in exclude]

corr = train[num_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values()

colors = ['#d73027' if v < 0 else '#4575b4' for v in corr]

fig, ax = plt.subplots(figsize=(7, 10))
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Pearson con log_price', fontsize=12)
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.show()

## 6. Scatter: área habitable vs precio

In [ ]:
sample = train.dropna(subset=['livingArea']).sample(2000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    sample['livingArea'],
    sample['lastSoldPrice_hpi_adjusted'],
    c=sample['log_price'], cmap='plasma',
    alpha=0.4, s=15, linewidths=0
)
ax.set_xlim(0, sample['livingArea'].quantile(0.99))
ax.set_ylim(0, sample['lastSoldPrice_hpi_adjusted'].quantile(0.99))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f} ft²'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_xlabel('Área habitable (ft²)')
ax.set_ylabel('Precio de venta')
ax.set_title('Área habitable vs Precio de venta (muestra 2,000 props)')
plt.colorbar(scatter, ax=ax, label='log_price')
plt.tight_layout()
plt.show()

## 7. Precio por número de habitaciones y baños

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col in zip(axes, ['bedrooms', 'bathrooms']):
    data = train[train[col].between(1, 6)].copy()
    data[col] = data[col].round(0)  # agrupar medios baños
    order = sorted(data[col].unique())
    groups = [data[data[col] == v]['lastSoldPrice_hpi_adjusted'].values for v in order]
    bp = ax.boxplot(groups, labels=[str(int(v)) for v in order],
                    patch_artist=True, showfliers=False,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.set_xlabel(col)
    ax.set_ylabel('Precio de venta')
    ax.set_title(f'Precio por cantidad de {col}')

plt.tight_layout()
plt.show()

## 8. Mapa geográfico de precios

In [ ]:
sample_geo = train.dropna(subset=['latitude', 'longitude']).sample(3000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 9))
sc = ax.scatter(
    sample_geo['longitude'],
    sample_geo['latitude'],
    c=sample_geo['log_price'],
    cmap='RdYlGn', alpha=0.5, s=10, linewidths=0
)
plt.colorbar(sc, ax=ax, label='log_price (verde = más caro)')
ax.set_title('Distribución geográfica de precios\n(muestra 3,000 props)', fontsize=12)
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
plt.tight_layout()
plt.show()

## 9. Features booleanos: impacto en precio

In [ ]:
bool_cols = ['has_pool', 'has_garage', 'has_waterfront', 'has_hoa',
             'tag_new_construction', 'tag_foreclosure', 'tag_price_cut']

results = []
for col in bool_cols:
    yes = train[train[col] == 1]['lastSoldPrice_hpi_adjusted'].median()
    no  = train[train[col] == 0]['lastSoldPrice_hpi_adjusted'].median()
    results.append({'feature': col, 'Con (mediana)': yes, 'Sin (mediana)': no,
                    'Diferencia %': round((yes - no) / no * 100, 1)})

df_bool = pd.DataFrame(results).set_index('feature')
df_bool[['Con (mediana)', 'Sin (mediana)']] = df_bool[['Con (mediana)', 'Sin (mediana)']].applymap(lambda x: f'${x:,.0f}')
df_bool

In [ ]:
diffs = []
for col in bool_cols:
    yes = train[train[col] == 1]['lastSoldPrice_hpi_adjusted'].median()
    no  = train[train[col] == 0]['lastSoldPrice_hpi_adjusted'].median()
    diffs.append((col, round((yes - no) / no * 100, 1)))

df_diff = pd.DataFrame(diffs, columns=['feature', 'diff_pct']).sort_values('diff_pct')
colors = ['#d73027' if v < 0 else '#4575b4' for v in df_diff['diff_pct']]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_diff['feature'], df_diff['diff_pct'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Diferencia % en precio mediano (con vs sin)')
ax.set_title('Impacto de features booleanos en el precio mediano')
plt.tight_layout()
plt.show()

## 10. Heatmap de correlaciones entre features numéricos

In [ ]:
cols_heatmap = [
    'log_price', 'livingArea', 'bedrooms', 'bathrooms', 'yearBuilt',
    'taxAssessedValue', 'last_listing_price', 'avg_school_rating',
    'hoa_fee_monthly', 'lotAreaValue', 'photoCount', 'property_age'
]

corr_matrix = train[cols_heatmap].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Heatmap de correlaciones (features seleccionados)', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Precio por ZIP code (top 20)

In [ ]:
top_zips = train['zipcode'].value_counts().head(20).index
zip_stats = (
    train[train['zipcode'].isin(top_zips)]
    .groupby('zipcode')['lastSoldPrice_hpi_adjusted']
    .median()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
zip_stats.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title('Precio mediano por ZIP code (top 20 más frecuentes)')
ax.set_xlabel('ZIP code')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 12. Conclusiones para el modelado

| Hallazgo | Implicancia |
|---|---|
| `log_price` tiene distribución aproximadamente normal | Entrenar en log-scale es correcto |
| `taxAssessedValue` y `latest_tax_value` correlacionan ~0.70 | Features leaky pero válidos para usar |
| `lotAreaValue` tiene 45% de missing | Imputar con mediana por homeType |
| `last_listing_price` tiene 33% de missing | Imputar o crear flag binario |
| `has_waterfront` tiene gran impacto en precio | Feature importante para el modelo |
| `tag_foreclosure` reduce el precio | Feature importante — captura ventas en apuros |
| Variación geográfica alta | ZIP code y lat/lon son predictores relevantes |
| SINGLE_FAMILY y CONDO dominan | Posible entrenar modelos separados por tipo |